# 05 · Feature engineering

Input: `data/interim/lendingclub_clean.parquet`
Output: `data/processed/train.parquet`, `data/processed/valid.parquet`, `data/processed/test.parquet` (0 null values), `data/processed/feature_config.json`, `models/preprocessor.joblib`

Scope: application-time features, leakage guard, chronological split, missingness audit, train-fitted imputation. Categorical encoding happens in the modelling pipeline.

In [14]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.3f}'.format)

TARGET = 'default'
SPLIT_KEY = 'issue_d'
TRAIN_END = pd.Timestamp('2013-01-01')
VALID_END = pd.Timestamp('2014-01-01')
TEST_END = pd.Timestamp('2015-01-01')

INTERIM_PATH = Path('../data/interim/lendingclub_clean.parquet')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(INTERIM_PATH)
print(f'Shape: {df.shape}')
print(f'Date range: {df[SPLIT_KEY].min():%b %Y} to {df[SPLIT_KEY].max():%b %Y}')
print(f'Default rate: {df[TARGET].mean():.2%}')

Shape: (453809, 73)
Date range: Jun 2007 to Dec 2014
Default rate: 17.03%


## 1. Leakage guard

In [15]:
POST_ORIGINATION = {
    'loan_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries',
    'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low',
    'pymnt_plan', 'hardship_flag', 'debt_settlement_flag',
}
POST_ORIGINATION_PREFIXES = ('hardship_', 'settlement_', 'debt_settlement')

leaky = sorted(
    c for c in df.columns
    if c != TARGET and (c in POST_ORIGINATION or c.startswith(POST_ORIGINATION_PREFIXES))
)
print('Post-origination columns removed:', leaky or 'none')
model_df = df.drop(columns=leaky)

Post-origination columns removed: none


## 2. Feature construction

In [22]:
year = model_df[SPLIT_KEY].dt.year

train_mask = model_df[SPLIT_KEY] < TRAIN_END
valid_mask = (
    (model_df[SPLIT_KEY] >= TRAIN_END)
    & (model_df[SPLIT_KEY] < VALID_END)
)
test_mask = (
    (model_df[SPLIT_KEY] >= VALID_END)
    & (model_df[SPLIT_KEY] < TEST_END)
)

print("Train rows:", train_mask.sum())
print("Validation rows:", valid_mask.sum())
print("Test rows:", test_mask.sum())

assert train_mask.any() and valid_mask.any() and test_mask.any()
assert (train_mask | valid_mask | test_mask).all(), "Rows outside the split window"
assert not (train_mask & valid_mask).any()
assert not (valid_mask & test_mask).any()

train_df = model_df.loc[train_mask].reset_index(drop=True)
valid_df = model_df.loc[valid_mask].reset_index(drop=True)
test_df = model_df.loc[test_mask].reset_index(drop=True)

cohort = model_df.groupby(year).agg(
    rows=(TARGET, "size"),
    default_rate=(TARGET, "mean"),
)

if "term" in model_df.columns:
    cohort["share_60m"] = model_df["term"].eq(60).groupby(year).mean()

display(cohort)

split_summary = pd.DataFrame(
    {
        "rows": [len(train_df), len(valid_df), len(test_df)],
        "start": [
            train_df[SPLIT_KEY].min(),
            valid_df[SPLIT_KEY].min(),
            test_df[SPLIT_KEY].min(),
        ],
        "end": [
            train_df[SPLIT_KEY].max(),
            valid_df[SPLIT_KEY].max(),
            test_df[SPLIT_KEY].max(),
        ],
        "default_rate": [
            train_df[TARGET].mean(),
            valid_df[TARGET].mean(),
            test_df[TARGET].mean(),
        ],
    },
    index=["train", "valid", "test"],
)

display(split_summary)

Train rows: 95902
Validation rows: 134804
Test rows: 223103


,rows,default_rate,share_60m
issue_d,,,
2007,603,0.262,0.000
2008,2393,0.207,0.000
2009,5281,0.137,0.000
2010,12537,0.140,0.270
2011,21721,0.152,0.351
2012,53367,0.162,0.185
2013,134804,0.156,0.255
2014,223103,0.184,0.271


,rows,start,end,default_rate
train,95902,2007-06-01,2012-12-01,0.157
valid,134804,2013-01-01,2013-12-01,0.156
test,223103,2014-01-01,2014-12-01,0.184


## 3. Chronological split

Train: loans issued before 2014. Validation: loans issued in 2014, used for model and threshold selection. Test: loans issued in 2015, scored once with the selected model and the frozen threshold.

In [24]:
year = model_df[SPLIT_KEY].dt.year
train_mask = model_df[SPLIT_KEY] < TRAIN_END
valid_mask = (model_df[SPLIT_KEY] >= TRAIN_END) & (model_df[SPLIT_KEY] < VALID_END)
test_mask = (model_df[SPLIT_KEY] >= VALID_END) & (model_df[SPLIT_KEY] < TEST_END)

assert train_mask.any() and valid_mask.any() and test_mask.any()
assert (train_mask | valid_mask | test_mask).all(), 'Rows outside the split window'
assert not (train_mask & valid_mask).any()
assert not (valid_mask & test_mask).any()

train_df = model_df.loc[train_mask].reset_index(drop=True)
valid_df = model_df.loc[valid_mask].reset_index(drop=True)
test_df = model_df.loc[test_mask].reset_index(drop=True)

cohort = model_df.groupby(year).agg(rows=(TARGET, 'size'), default_rate=(TARGET, 'mean'))
if 'term' in model_df.columns:
    cohort['share_60m'] = model_df['term'].eq(60).groupby(year).mean()
display(cohort)

split_summary = pd.DataFrame({
    'rows': [len(train_df), len(valid_df), len(test_df)],
    'start': [train_df[SPLIT_KEY].min(), valid_df[SPLIT_KEY].min(), test_df[SPLIT_KEY].min()],
    'end': [train_df[SPLIT_KEY].max(), valid_df[SPLIT_KEY].max(), test_df[SPLIT_KEY].max()],
    'default_rate': [train_df[TARGET].mean(), valid_df[TARGET].mean(), test_df[TARGET].mean()],
}, index=['train', 'valid', 'test'])
split_summary

,rows,default_rate,share_60m
issue_d,,,
2007,603,0.262,0.000
2008,2393,0.207,0.000
2009,5281,0.137,0.000
2010,12537,0.140,0.270
2011,21721,0.152,0.351
2012,53367,0.162,0.185
2013,134804,0.156,0.255
2014,223103,0.184,0.271


,rows,start,end,default_rate
train,95902,2007-06-01,2012-12-01,0.157
valid,134804,2013-01-01,2013-12-01,0.156
test,223103,2014-01-01,2014-12-01,0.184


## 4. Missingness audit

In [18]:
INDICATOR_COLS = [c for c in FEATURES if c.endswith('_missing')]
for ind in INDICATOR_COLS:
    base = ind.removesuffix('_missing')
    assert base in FEATURES, f'{ind} has no source column'
    for name, part in (('train', train_df), ('valid', valid_df), ('test', test_df)):
        same = part[ind].astype('int8').eq(part[base].isna().astype('int8')).all()
        assert same, f'{ind} does not match {base} missingness in {name}'

audit = pd.DataFrame({
    'train_missing_%': train_df[FEATURES].isna().mean() * 100,
    'valid_missing_%': valid_df[FEATURES].isna().mean() * 100,
    'test_missing_%': test_df[FEATURES].isna().mean() * 100,
})
audit = audit[audit['train_missing_%'] > 0].sort_values('train_missing_%', ascending=False)
audit['has_indicator'] = [f'{c}_missing' in FEATURES for c in audit.index]
audit

,train_missing_%,valid_missing_%,test_missing_%,has_indicator
mths_since_last_record,94.656,88.132,82.089,True
mths_since_last_major_derog,94.387,79.874,71.532,True
mths_since_recent_bc_dlq,93.004,77.717,73.505,True
mths_since_recent_revol_delinq,85.489,69.936,63.997,True
mo_sin_old_il_acct,74.505,3.676,3.093,False
mo_sin_rcnt_rev_tl_op,73.280,0.000,0.000,False
mo_sin_old_rev_tl_op,73.280,0.000,0.000,False
num_bc_tl,73.279,0.000,0.000,False
total_rev_hi_lim,73.279,0.000,0.000,False
mo_sin_rcnt_tl,73.279,0.000,0.000,False


In [19]:
high_missing = audit.index[audit['train_missing_%'] >= 15].tolist()
(
    train_df[high_missing].isna()
    .groupby(train_df[SPLIT_KEY].dt.year).mean().mul(100).T
)

issue_d,2007,2008,2009,2010,2011,2012
mths_since_last_record,6.965,60.635,95.209,94.002,94.738,97.238
mths_since_last_major_derog,100.000,100.000,100.000,100.000,100.000,89.913
mths_since_recent_bc_dlq,100.000,100.000,100.000,100.000,100.000,87.429
mths_since_recent_revol_delinq,100.000,100.000,100.000,100.000,100.000,73.924
mo_sin_old_il_acct,100.000,100.000,100.000,100.000,100.000,54.185
mo_sin_rcnt_rev_tl_op,100.000,100.000,100.000,100.000,100.000,51.983
mo_sin_old_rev_tl_op,100.000,100.000,100.000,100.000,100.000,51.983
num_bc_tl,100.000,100.000,100.000,100.000,100.000,51.982
total_rev_hi_lim,100.000,100.000,100.000,100.000,100.000,51.982
mo_sin_rcnt_tl,100.000,100.000,100.000,100.000,100.000,51.982


## 5. Column groups and preprocessing check

Groups are written to `feature_config.json` so the modelling pipeline builds the same preprocessor. Event-history fields (`mths_since_*` with an indicator) are kept as a separate group: a missing value means no event on record or not collected in that period, so a median fill would invent an event.

In [20]:
X_train, X_valid, X_test = train_df[FEATURES], valid_df[FEATURES], test_df[FEATURES]

numeric_cols = X_train.select_dtypes(include='number').columns.tolist()
FLAGGED_COLS = [c.removesuffix('_missing') for c in INDICATOR_COLS]
EVENT_HISTORY_COLS = [c for c in FLAGGED_COLS if c.startswith('mths_since_')]
MODE_COLS = [c for c in ['emp_length'] if c in numeric_cols]
MEDIAN_COLS = [c for c in numeric_cols if c not in set(MODE_COLS) | set(EVENT_HISTORY_COLS)]
CATEGORICAL_COLS = [c for c in FEATURES if c not in numeric_cols]

assert set(EVENT_HISTORY_COLS) <= set(numeric_cols)
assert sorted(MEDIAN_COLS + MODE_COLS + EVENT_HISTORY_COLS + CATEGORICAL_COLS) == sorted(FEATURES)

GROUPS = {
    'numeric_median': MEDIAN_COLS,
    'numeric_mode': MODE_COLS,
    'event_history': EVENT_HISTORY_COLS,
    'categorical': CATEGORICAL_COLS,
}


def build_preprocessor(groups):
    return ColumnTransformer(
        [
            ('numeric_median', SimpleImputer(strategy='median', keep_empty_features=True), groups['numeric_median']),
            ('numeric_mode', SimpleImputer(strategy='most_frequent', keep_empty_features=True), groups['numeric_mode']),
            ('event_history', SimpleImputer(strategy='constant', fill_value=-1), groups['event_history']),
            ('categorical', SimpleImputer(strategy='constant', fill_value='Missing'), groups['categorical']),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    ).set_output(transform='pandas')


preprocessor = build_preprocessor(GROUPS)
X_train_pp = preprocessor.fit_transform(X_train)
X_valid_pp = preprocessor.transform(X_valid)
X_test_pp = preprocessor.transform(X_test)

assert list(X_train_pp.columns) == list(X_valid_pp.columns) == list(X_test_pp.columns)
assert set(X_train_pp.columns) == set(FEATURES), 'A feature was lost or added by the preprocessor'
assert X_train_pp.isna().sum().sum() == 0 and X_valid_pp.isna().sum().sum() == 0 and X_test_pp.isna().sum().sum() == 0
assert set(X_train_pp.select_dtypes(exclude='number').columns) == set(CATEGORICAL_COLS)

print('Preprocessed shapes:', X_train_pp.shape, X_valid_pp.shape, X_test_pp.shape)
print({name: len(cols) for name, cols in GROUPS.items()})

Preprocessed shapes: (95902, 69) (134804, 69) (223103, 69)
{'numeric_median': 58, 'numeric_mode': 1, 'event_history': 5, 'categorical': 5}


## 6. Save clean train, validation and test sets

In [21]:
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_out = pd.concat([train_df[[SPLIT_KEY, TARGET]], X_train_pp], axis=1)
valid_out = pd.concat([valid_df[[SPLIT_KEY, TARGET]], X_valid_pp], axis=1)
test_out = pd.concat([test_df[[SPLIT_KEY, TARGET]], X_test_pp], axis=1)

assert train_out.isna().sum().sum() == 0, 'train still has nulls'
assert valid_out.isna().sum().sum() == 0, 'valid still has nulls'
assert test_out.isna().sum().sum() == 0, 'test still has nulls'
assert list(train_out.columns) == list(valid_out.columns) == list(test_out.columns)

train_out.to_parquet(PROCESSED_DIR / 'train.parquet', index=False)
valid_out.to_parquet(PROCESSED_DIR / 'valid.parquet', index=False)
test_out.to_parquet(PROCESSED_DIR / 'test.parquet', index=False)
joblib.dump(preprocessor, MODEL_DIR / 'preprocessor.joblib')

config = {
    'target': TARGET,
    'split_key': SPLIT_KEY,
    'train_end': TRAIN_END.date().isoformat(),
    'valid_end': VALID_END.date().isoformat(),
    'test_end': TEST_END.date().isoformat(),
    'features': [c for c in train_out.columns if c not in {TARGET, SPLIT_KEY}],
    **GROUPS,
    'missing_indicators': INDICATOR_COLS,
    'rows': {'train': len(train_out), 'valid': len(valid_out), 'test': len(test_out)},
}
(PROCESSED_DIR / 'feature_config.json').write_text(json.dumps(config, indent=2))

for name, frame in (('train', train_out), ('valid', valid_out), ('test', test_out)):
    back = pd.read_parquet(PROCESSED_DIR / f'{name}.parquet')
    assert back.shape == frame.shape and list(back.columns) == list(frame.columns)
    assert back.isna().sum().sum() == 0
    print(f'{name}.parquet  {back.shape}  nulls: {back.isna().sum().sum()}  default rate {back[TARGET].mean():.2%}')

train.parquet  (95902, 71)  nulls: 0  default rate 15.72%
valid.parquet  (134804, 71)  nulls: 0  default rate 15.60%
test.parquet  (223103, 71)  nulls: 0  default rate 18.45%


### Missing-Value Handling Strategy

I handle missing values according to the column type, missingness rate, and meaning of the missing data.

| Group | Columns | Missingness | Fill Strategy |
|---|---|---|---|
| `numeric_median` | All other numeric columns | 0.01% to approximately 30% | Train-set median |
| `numeric_mode` | `emp_length` | Approximately 6.5% | Train-set most-frequent value, preserving its ordinal categories |
| `event_history` | `mths_since_*` columns that already have a corresponding `_missing` flag | Approximately 58% to 91% | Fill with `-1` and retain the existing missingness flag |
| `categorical` | Text/categorical columns | Any missingness rate | Introduce a new `"Missing"` category |

**Rationale**

- **Numeric columns:** Median imputation provides a robust baseline that is less sensitive to outliers than mean imputation. The median must be learned from the training set only and then applied to validation and test data.
- **`emp_length`:** Use the most-frequent training-set value as a simple baseline. Although employment length is ordinal, mode imputation preserves a valid category rather than introducing an artificial value.
- **Event-history columns:** A value of `-1` acts as a sentinel for missing historical information, while the existing `_missing` flag explicitly identifies missingness. This allows the model to distinguish unavailable history from observed values. Ensure that `-1` cannot be confused with a valid value.
- **Categorical columns:** The `"Missing"` category preserves missingness as a potentially informative signal instead of replacing it with an existing category.

**Implementation note:** Fit all imputation values using the training data only. Apply the learned values consistently to validation, test, and future data to prevent data leakage.